# E3 only: full-coverage reconstructed BigVul inputs

Train the all-12-layer UniXcoder + AST/CFG RGCN configuration from `f9867245`, seed 42, 10 epochs, batch 64, accumulation 8.

The rebuilt BigVul CSV and evaluation cohorts match the historical inputs. All 77,300 functions now have AST/CFG graph tensors: 5,313 exact source-hash cache hits and 71,987 fresh Joern 4.0.506 exports from a reused JVM. The rebuilt vocabulary has 61,763 node labels; the historical checkpoint had 2,822 node embeddings, and its original vocabulary mapping and graph-presence mask were not retained. Four CWE vectors were regenerated. This is a **fresh E3 run on full-coverage reconstructed features**, not proof of the historical result. Report only metrics produced by this run.

Upload `e3-inputs-full-joern-20261014.zip` to `My Drive/data/graph-vul/e3-reproduction-20261014/`. Select a GPU runtime. Every execution writes a new run folder; previous source files and results are preserved.


In [ ]:
%pip install -q torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu128
%pip install -q numpy==2.2.6 pandas==2.3.3 scikit-learn==1.7.2 transformers==5.3.0 pytorch-lightning==2.6.1 torchmetrics==1.9.0 torch-geometric==2.7.0 sentencepiece==0.2.1 wandb==0.25.1 tqdm==4.67.3 matplotlib==3.10.8 seaborn==0.13.2


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import os, subprocess, sys, zipfile

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/data/graph-vul/e3-reproduction-20261014")
BUNDLE = DRIVE_ROOT / "e3-inputs-full-joern-20261014.zip"
STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
LOCAL = Path("/content") / ("e3-inputs-" + STAMP)
LOCAL.mkdir(exist_ok=False)
with zipfile.ZipFile(BUNDLE) as archive:
    archive.extractall(LOCAL)
PREPARED = LOCAL / "prepared"
print("Prepared inputs:", PREPARED)


In [ ]:
REPO = Path("/content") / ("graph-vul-e3-" + STAMP)
subprocess.run(["git", "clone", "--branch", "portfolio/appsflyer", "--single-branch", "https://github.com/noamdwc/graph-vul.git", str(REPO)], check=True)
RUNNER = REPO / "scripts/run_e3_reproduction.py"
print("Source revision:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())


In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before launching E3"
print(torch.cuda.get_device_name(0))
OUTPUT = DRIVE_ROOT / "runs" / ("E3_all_layers-" + STAMP)
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
print("Fresh output:", OUTPUT)
env = os.environ.copy()
env["WANDB_MODE"] = "disabled"
subprocess.run([sys.executable, str(RUNNER), "--data-root", str(PREPARED), "--output-dir", str(OUTPUT)], cwd=REPO, env=env, check=True)


In [ ]:
import json, pandas as pd
metrics = json.loads((OUTPUT / "metrics.json").read_text())
display(pd.DataFrame.from_dict(metrics, orient="index"))
print("Keep the entire run directory, including checkpoint, predictions, and run.json:", OUTPUT)
